In [3]:
# ============================================================
# 1. IMPORTAR LIBRERÍAS
# ============================================================
import json
import pandas as pd
from collections import Counter
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

# ============================================================
# 2. CONFIGURACIÓN
# ============================================================
file_path = r'C:\Users\Aaron\Documents\arxiv_cs_dataset_exploration\dataset\arxiv-metadata-oai-snapshot.json'

# leer todo el archivo: sample_size = None
sample_size = None

# ============================================================
# 3. CARGAR EL DATASET JSONL
# ============================================================
# Este archivo de arXiv está en formato JSON Lines:
# cada línea es un JSON independiente

records = []

with open(file_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        record = json.loads(line)
        records.append(record)
        
        if sample_size is not None and i + 1 >= sample_size:
            break

df = pd.DataFrame(records)

print("Cantidad de registros cargados:", len(df))
print("Dimensiones:", df.shape)
df.head()

Cantidad de registros cargados: 2982054
Dimensiones: (2982054, 14)


,id,submitter,authors,title,comments,journal-ref,doi,report-no,categories,license,abstract,versions,update_date,authors_parsed
0,0704.0001,Pavel Nadolsky,"C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-P. Yuan",Calculation of prompt diphoton production cross sections at Tevatron and\n LHC energies,"37 pages, 15 figures; published version","Phys.Rev.D76:013009,2007",10.1103/PhysRevD.76.013009,ANL-HEP-PR-07-12,hep-ph,NaN,A fully differential calculation in perturbative quantum chromodynamics is\npresented for the production of massive photon pairs at hadron colliders. All\nnext-to-leading order perturbative cont...,"[{'version': 'v1', 'created': 'Mon, 2 Apr 2007 19:18:42 GMT'}, {'version': 'v2', 'created': 'Tue, 24 Jul 2007 20:10:27 GMT'}]",2008-11-26,"[[Balázs, C., ], [Berger, E. L., ], [Nadolsky, P. M., ], [Yuan, C. -P., ]]"
1,0704.0002,Louis Theran,Ileana Streinu and Louis Theran,Sparsity-certifying Graph Decompositions,To appear in Graphs and Combinatorics,NaN,NaN,NaN,math.CO cs.CG,http://arxiv.org/licenses/nonexclusive-distrib/1.0/,"We describe a new algorithm, the $(k,\ell)$-pebble game with colors, and use\nit obtain a characterization of the family of $(k,\ell)$-sparse graphs and\nalgorithmic solutions to a family of pro...","[{'version': 'v1', 'created': 'Sat, 31 Mar 2007 02:26:18 GMT'}, {'version': 'v2', 'created': 'Sat, 13 Dec 2008 17:26:00 GMT'}]",2008-12-13,"[[Streinu, Ileana, ], [Theran, Louis, ]]"
2,0704.0003,Hongjun Pan,Hongjun Pan,The evolution of the Earth-Moon system based on the dark matter field\n fluid model,"23 pages, 3 figures",NaN,NaN,NaN,physics.gen-ph,NaN,"The evolution of Earth-Moon system is described by the dark matter field\nfluid model proposed in the Meeting of Division of Particle and Field 2004,\nAmerican Physical Society. The current beha...","[{'version': 'v1', 'created': 'Sun, 1 Apr 2007 20:46:54 GMT'}, {'version': 'v2', 'created': 'Sat, 8 Dec 2007 23:47:24 GMT'}, {'version': 'v3', 'created': 'Sun, 13 Jan 2008 00:36:28 GMT'}]",2008-01-13,"[[Pan, Hongjun, ]]"
3,0704.0004,David Callan,David Callan,A determinant of Stirling cycle numbers counts unlabeled acyclic\n single-source automata,11 pages,NaN,NaN,NaN,math.CO,NaN,We show that a determinant of Stirling cycle numbers counts unlabeled acyclic\nsingle-source automata. The proof involves a bijection from these automata to\ncertain marked lattice paths and a s...,"[{'version': 'v1', 'created': 'Sat, 31 Mar 2007 03:16:14 GMT'}]",2007-05-23,"[[Callan, David, ]]"
4,0704.0005,Alberto Torchinsky,Wael Abu-Shammala and Alberto Torchinsky,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\alpha}$,NaN,"Illinois J. Math. 52 (2008) no.2, 681-689",NaN,NaN,math.CA math.FA,NaN,"In this paper we show how to compute the $\Lambda_{\alpha}$ norm, $\alpha\ge\n0$, using the dyadic grid. This result is a consequence of the description of\nthe Hardy spaces $H^p(R^N)$ in terms ...","[{'version': 'v1', 'created': 'Mon, 2 Apr 2007 18:09:58 GMT'}]",2013-10-15,"[[Abu-Shammala, Wael, ], [Torchinsky, Alberto, ]]"


In [4]:
# ============================================================
# 4. EXPLORACIÓN GENERAL DEL DATASET
# ============================================================
print("Columnas del dataset:")
print(df.columns.tolist())

Columnas del dataset:
['id', 'submitter', 'authors', 'title', 'comments', 'journal-ref', 'doi', 'report-no', 'categories', 'license', 'abstract', 'versions', 'update_date', 'authors_parsed']


In [5]:
# Información general
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2982054 entries, 0 to 2982053
Data columns (total 14 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   id              str   
 1   submitter       str   
 2   authors         str   
 3   title           str   
 4   comments        str   
 5   journal-ref     str   
 6   doi             str   
 7   report-no       str   
 8   categories      str   
 9   license         str   
 10  abstract        str   
 11  versions        object
 12  update_date     str   
 13  authors_parsed  object
dtypes: object(2), str(12)
memory usage: 318.5+ MB


In [6]:
# Valores faltantes por columna
missing = df.isnull().sum().sort_values(ascending=False)
missing

report-no         2792092
journal-ref       2045980
doi               1685663
comments           814949
license            452752
submitter           15185
authors                 0
title                   0
id                      0
categories              0
abstract                0
versions                0
update_date             0
authors_parsed          0
dtype: int64

In [7]:
# Vista rápida de algunas columnas clave
columns_of_interest = ["id", "title", "authors", "categories", "abstract", "journal-ref", "doi"]
df[columns_of_interest].head(3)

,id,title,authors,categories,abstract,journal-ref,doi
0,0704.0001,Calculation of prompt diphoton production cross sections at Tevatron and\n LHC energies,"C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-P. Yuan",hep-ph,A fully differential calculation in perturbative quantum chromodynamics is\npresented for the production of massive photon pairs at hadron colliders. All\nnext-to-leading order perturbative cont...,"Phys.Rev.D76:013009,2007",10.1103/PhysRevD.76.013009
1,0704.0002,Sparsity-certifying Graph Decompositions,Ileana Streinu and Louis Theran,math.CO cs.CG,"We describe a new algorithm, the $(k,\ell)$-pebble game with colors, and use\nit obtain a characterization of the family of $(k,\ell)$-sparse graphs and\nalgorithmic solutions to a family of pro...",NaN,NaN
2,0704.0003,The evolution of the Earth-Moon system based on the dark matter field\n fluid model,Hongjun Pan,physics.gen-ph,"The evolution of Earth-Moon system is described by the dark matter field\nfluid model proposed in the Meeting of Division of Particle and Field 2004,\nAmerican Physical Society. The current beha...",NaN,NaN


In [8]:
# ============================================================
# 5. LONGITUD DE TÍTULOS Y ABSTRACTS
# ============================================================
df["title_len"] = df["title"].fillna("").apply(len)
df["abstract_len"] = df["abstract"].fillna("").apply(len)

print("Resumen de longitud de títulos:")
display(df["title_len"].describe())

print("Resumen de longitud de abstracts:")
display(df["abstract_len"].describe())

Resumen de longitud de títulos:


count    2.982054e+06
mean     7.557926e+01
std      2.748670e+01
min      1.000000e+00
25%      5.600000e+01
50%      7.200000e+01
75%      9.200000e+01
max      3.820000e+02
Name: title_len, dtype: float64

Resumen de longitud de abstracts:


count    2.982054e+06
mean     9.936401e+02
std      4.330538e+02
min      3.000000e+00
25%      6.570000e+02
50%      9.730000e+02
75%      1.309000e+03
max      6.091000e+03
Name: abstract_len, dtype: float64

In [11]:
# ============================================================
# 6. EXPLORACIÓN DE CATEGORÍAS
# ============================================================
# La columna 'categories' contiene una o varias categorías separadas por espacio

df["categories"] = df["categories"].fillna("")
df["category_list"] = df["categories"].apply(lambda x: x.split())

# Categoría principal: la primera de la lista
df["main_category"] = df["category_list"].apply(lambda x: x[0] if len(x) > 0 else None)

df[["categories", "category_list", "main_category"]]

,categories,category_list,main_category
0,hep-ph,[hep-ph],hep-ph
1,math.CO cs.CG,"[math.CO, cs.CG]",math.CO
2,physics.gen-ph,[physics.gen-ph],physics.gen-ph
3,math.CO,[math.CO],math.CO
4,math.CA math.FA,"[math.CA, math.FA]",math.CA
...,...,...,...
2982049,supr-con cond-mat.supr-con,"[supr-con, cond-mat.supr-con]",supr-con
2982050,supr-con cond-mat.supr-con,"[supr-con, cond-mat.supr-con]",supr-con
2982051,supr-con cond-mat.supr-con,"[supr-con, cond-mat.supr-con]",supr-con
2982052,supr-con cond-mat.supr-con,"[supr-con, cond-mat.supr-con]",supr-con


In [10]:
# Conteo de categorías principales
main_cat_counts = df["main_category"].value_counts(dropna=False)
main_cat_counts.head(30)

main_category
cs.CV                 142953
hep-ph                141480
quant-ph              128185
cs.LG                 127899
hep-th                112079
astro-ph               94246
cs.CL                  77738
gr-qc                  70608
cond-mat.mtrl-sci      69606
cond-mat.mes-hall      68640
math.AP                56477
astro-ph.GA            53799
math.CO                52165
cond-mat.str-el        52082
astro-ph.SR            47682
astro-ph.HE            45299
astro-ph.CO            44302
math.PR                43383
cond-mat.stat-mech     43284
math.AG                39575
cs.IT                  39054
math.OC                38011
cs.RO                  37322
math.NT                36663
cs.AI                  35511
nucl-th                35404
physics.optics         34854
math.NA                34305
math-ph                33839
cond-mat.supr-con      33763
Name: count, dtype: int64

In [12]:
# Conteo de TODAS las categorías, no solo la principal
all_categories_counter = Counter()

for cat_list in df["category_list"]:
    all_categories_counter.update(cat_list)

all_categories_df = pd.DataFrame(
    all_categories_counter.items(),
    columns=["category", "count"]
).sort_values("count", ascending=False).reset_index(drop=True)

all_categories_df.head(50)

,category,count
0,cs.LG,257067
1,hep-ph,194395
2,cs.CV,184612
3,hep-th,180625
4,quant-ph,175671
5,cs.AI,166910
6,gr-qc,120089
7,cond-mat.mtrl-sci,107107
8,astro-ph,105380
9,cs.CL,104104


In [13]:
print("Número total de categorías distintas:", all_categories_df["category"].nunique())

Número total de categorías distintas: 176


In [ ]:
# ============================================================
# 7. IDENTIFICAR PREFIJOS DE ÁREAS
# ============================================================
# En arXiv, muchas categorías tienen prefijos como:
# cs.  -> Computer Science
# math -> Mathematics
# physics / astro-ph / quant-ph / hep-* -> Physics related
# stat -> Statistics
# q-bio -> Quantitative Biology
# q-fin -> Quantitative Finance
# econ -> Economics

In [14]:
def get_area_prefix(category):
    if pd.isna(category) or category is None:
        return None
    
    # Casos especiales de arXiv
    special_prefixes = [
        "astro-ph", "cond-mat", "gr-qc", "hep-ex", "hep-lat", "hep-ph", "hep-th",
        "math-ph", "nlin", "nucl-ex", "nucl-th", "quant-ph", "q-bio", "q-fin", "cmp-lg"
    ]
    
    for prefix in special_prefixes:
        if category.startswith(prefix):
            return prefix
    
    # Regla general: tomar lo que está antes del primer punto
    return category.split(".")[0]

all_categories_df["area_prefix"] = all_categories_df["category"].apply(get_area_prefix)
all_categories_df.head(30)

,category,count,area_prefix
0,cs.LG,257067,cs
1,hep-ph,194395,hep-ph
2,cs.CV,184612,cs
3,hep-th,180625,hep-th
4,quant-ph,175671,quant-ph
5,cs.AI,166910,cs
6,gr-qc,120089,gr-qc
7,cond-mat.mtrl-sci,107107,cond-mat
8,astro-ph,105380,astro-ph
9,cs.CL,104104,cs


In [15]:
# Resumen por área/prefijo
area_counts = all_categories_df.groupby("area_prefix")["count"].sum().sort_values(ascending=False)
area_counts

area_prefix
cs          1358770
math        1011227
cond-mat     541770
astro-ph     462628
physics      363714
hep-ph       194395
hep-th       180625
quant-ph     175671
stat         172293
eess         125524
gr-qc        120089
math-ph       88659
nucl-th       61896
q-bio         61601
hep-ex        59453
nlin          50031
hep-lat       30076
q-fin         29581
nucl-ex       28190
econ          15237
chao-dyn       2398
q-alg          1578
alg-geom       1423
solv-int       1413
cmp-lg          894
dg-ga           732
patt-sol        650
adap-org        584
funct-an        427
mtrl-th         262
chem-ph         251
comp-gas        221
supr-con        175
atom-ph         123
acc-phys         49
plasm-ph         38
ao-sci           17
bayes-an         16
Name: count, dtype: int64

In [16]:
# ============================================================
# 8. ÉNFASIS EN CIENCIAS DE LA COMPUTACIÓN Y ÁREAS EXACTAS
# ============================================================
# Vamos a definir grupos de interés
computer_science_prefixes = ["cs"]
mathematics_prefixes = ["math", "math-ph"]
physics_prefixes = [
    "physics", "astro-ph", "cond-mat", "gr-qc", "hep-ex", "hep-lat",
    "hep-ph", "hep-th", "nucl-ex", "nucl-th", "quant-ph"
]
statistics_prefixes = ["stat"]
other_exact_sciences_prefixes = ["nlin"]  # nonlinear sciences

exact_sciences_prefixes = (
    mathematics_prefixes +
    physics_prefixes +
    statistics_prefixes +
    other_exact_sciences_prefixes
)

print("Prefijos de Computer Science:", computer_science_prefixes)
print("Prefijos de Ciencias Exactas:", exact_sciences_prefixes)

Prefijos de Computer Science: ['cs']
Prefijos de Ciencias Exactas: ['math', 'math-ph', 'physics', 'astro-ph', 'cond-mat', 'gr-qc', 'hep-ex', 'hep-lat', 'hep-ph', 'hep-th', 'nucl-ex', 'nucl-th', 'quant-ph', 'stat', 'nlin']


In [17]:
# Ver categorías específicas de ciencias de la computación
cs_categories_df = all_categories_df[all_categories_df["category"].str.startswith("cs.", na=False)].copy()
cs_categories_df

,category,count,area_prefix
0,cs.LG,257067,cs
2,cs.CV,184612,cs
5,cs.AI,166910,cs
9,cs.CL,104104,cs
28,cs.IT,53429,cs
30,cs.RO,50543,cs
35,cs.CR,46353,cs
37,cs.SY,43769,cs
46,cs.NA,33202,cs
50,cs.HC,28878,cs


In [18]:
print("Número de categorías distintas de Computer Science:", len(cs_categories_df))

Número de categorías distintas de Computer Science: 40


In [19]:
# Ver categorías de matemáticas
math_categories_df = all_categories_df[
    all_categories_df["category"].str.startswith("math", na=False)
].copy()

math_categories_df.head(50)

,category,count,area_prefix
11,math.MP,88659,math
12,math-ph,88659,math-ph
15,math.CO,76737,math
19,math.AP,72520,math
22,math.PR,64534,math
24,math.OC,59795,math
26,math.AG,58666,math
29,math.IT,53429,math
31,math.NT,48084,math
32,math.DG,47096,math


In [20]:
# Ver categorías relacionadas con física
physics_categories_df = all_categories_df[
    all_categories_df["area_prefix"].isin(physics_prefixes)
].copy()

physics_categories_df.head(100)

,category,count,area_prefix
1,hep-ph,194395,hep-ph
3,hep-th,180625,hep-th
4,quant-ph,175671,quant-ph
6,gr-qc,120089,gr-qc
7,cond-mat.mtrl-sci,107107,cond-mat
8,astro-ph,105380,astro-ph
10,cond-mat.mes-hall,100059,cond-mat
13,cond-mat.str-el,81789,cond-mat
14,cond-mat.stat-mech,80429,cond-mat
17,astro-ph.GA,75959,astro-ph


In [21]:
# ============================================================
# 9. CLASIFICAR CADA PAPER POR GRAN ÁREA
# ============================================================
def assign_general_area(cat_list):
    if not cat_list:
        return "Sin categoría"
    
    prefixes = [get_area_prefix(cat) for cat in cat_list]
    
    if any(prefix in computer_science_prefixes for prefix in prefixes):
        return "Computer Science"
    elif any(prefix in mathematics_prefixes for prefix in prefixes):
        return "Mathematics"
    elif any(prefix in physics_prefixes for prefix in prefixes):
        return "Physics"
    elif any(prefix in statistics_prefixes for prefix in prefixes):
        return "Statistics"
    elif any(prefix in other_exact_sciences_prefixes for prefix in prefixes):
        return "Other Exact Sciences"
    else:
        return "Other"

df["general_area"] = df["category_list"].apply(assign_general_area)

df["general_area"].value_counts()

general_area
Physics                 1378719
Computer Science         892992
Mathematics              606702
Other                     51661
Statistics                37082
Other Exact Sciences      14898
Name: count, dtype: int64

In [22]:
# Ver algunos ejemplos por área
for area in df["general_area"].dropna().unique():
    print(f"\n==================== {area} ====================")
    display(df[df["general_area"] == area][["id", "title", "categories"]].head(3))


==================== Physics ====================


,id,title,categories
0,0704.0001,Calculation of prompt diphoton production cross sections at Tevatron and\n LHC energies,hep-ph
2,0704.0003,The evolution of the Earth-Moon system based on the dark matter field\n fluid model,physics.gen-ph
5,0704.0006,Bosonic characters of atomic Cooper pairs across resonance,cond-mat.mes-hall



==================== Computer Science ====================


,id,title,categories
1,0704.0002,Sparsity-certifying Graph Decompositions,math.CO cs.CG
45,0704.0046,A limit relation for entropy and channel capacity per unit cost,quant-ph cs.IT math.IT
46,0704.0047,Intelligent location of simultaneously active acoustic emission sources:\n Part I,cs.NE cs.AI



==================== Mathematics ====================


...

,id,title,categories
23,0704.0024,Formation of quasi-solitons in transverse confined ferromagnetic film\n media,nlin.PS
44,0704.0045,Evolution of solitary waves and undular bores in shallow-water flows\n over a gradual slope with bottom friction,nlin.PS nlin.SI
95,0704.0096,Much ado about 248,nlin.SI



==================== Other ====================


,id,title,categories
35,0704.0036,A remark on the number of steady states in a multiple futile cycle,q-bio.QM q-bio.MN
157,0704.0158,Complexities of Human Promoter Sequences,q-bio.OT
330,0704.0331,Symmetries by base substitutions in the genetic code predict 2' or 3'\n aminoacylation of tRNAs,q-bio.OT



==================== Statistics ====================


,id,title,categories
1710,0704.1711,"Dynamical Equilibrium, trajectories study in an economical system. The\n case of the labor market",stat.AP
3473,0704.3474,Missing Data: A Comparison of Neural Network and Expectation\n Maximisation Techniques,stat.AP
3861,0704.3862,An Integrated Human-Computer System for Controlling Interstate Disputes,stat.AP


In [23]:
# ============================================================
# 10. CÓMO EXTRAER REGISTROS DE CIENCIAS DE LA COMPUTACIÓN
# ============================================================
# Un paper será considerado de CS si al menos una de sus categorías empieza con 'cs.'

def has_cs_category(cat_list):
    return any(cat.startswith("cs.") for cat in cat_list)

df_cs = df[df["category_list"].apply(has_cs_category)].copy()

print("Cantidad de papers de Computer Science:", len(df_cs))
df_cs[["id", "title", "categories", "main_category"]].head()

Cantidad de papers de Computer Science: 892992


,id,title,categories,main_category
1,0704.0002,Sparsity-certifying Graph Decompositions,math.CO cs.CG,math.CO
45,0704.0046,A limit relation for entropy and channel capacity per unit cost,quant-ph cs.IT math.IT,quant-ph
46,0704.0047,Intelligent location of simultaneously active acoustic emission sources:\n Part I,cs.NE cs.AI,cs.NE
49,0704.0050,Intelligent location of simultaneously active acoustic emission sources:\n Part II,cs.NE cs.AI,cs.NE
61,0704.0062,On-line Viterbi Algorithm and Its Relationship to Random Walks,cs.DS,cs.DS


In [24]:
# ============================================================
# 11. CÓMO EXTRAER REGISTROS DE CIENCIAS EXACTAS
# ============================================================
# Consideraremos aquí matemáticas, física, estadística y nlin

def has_exact_science_category(cat_list):
    prefixes = [get_area_prefix(cat) for cat in cat_list]
    return any(prefix in exact_sciences_prefixes for prefix in prefixes)

df_exact = df[df["category_list"].apply(has_exact_science_category)].copy()

print("Cantidad de papers de ciencias exactas:", len(df_exact))
df_exact[["id", "title", "categories", "main_category"]].head()

Cantidad de papers de ciencias exactas: 2271638


,id,title,categories,main_category
0,0704.0001,Calculation of prompt diphoton production cross sections at Tevatron and\n LHC energies,hep-ph,hep-ph
1,0704.0002,Sparsity-certifying Graph Decompositions,math.CO cs.CG,math.CO
2,0704.0003,The evolution of the Earth-Moon system based on the dark matter field\n fluid model,physics.gen-ph,physics.gen-ph
3,0704.0004,A determinant of Stirling cycle numbers counts unlabeled acyclic\n single-source automata,math.CO,math.CO
4,0704.0005,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\alpha}$,math.CA math.FA,math.CA


In [25]:
# ============================================================
# 12. COMPARACIÓN ENTRE COMPUTER SCIENCE Y CIENCIAS EXACTAS
# ============================================================
summary_df = pd.DataFrame({
    "subset": ["Computer Science", "Exact Sciences"],
    "rows": [len(df_cs), len(df_exact)]
})

summary_df

,subset,rows
0,Computer Science,892992
1,Exact Sciences,2271638


In [26]:
# Categorías principales más frecuentes dentro de CS
df_cs["main_category"].value_counts().head(30)

main_category
cs.CV       142953
cs.LG       127899
cs.CL        77738
cs.IT        39054
cs.RO        37322
cs.AI        35511
cs.CR        31310
math.NA      23047
eess.SY      20491
cs.SE        19519
cs.NI        18857
cs.HC        17892
cs.DS        17786
stat.ML      17786
cs.DC        16596
eess.IV      15463
cs.CY        14608
cs.IR        14312
cs.LO        11841
cs.SI        11629
math.OC      10634
cs.SD         9522
quant-ph      9139
cs.GT         9088
cs.NE         8012
eess.SP       7991
cs.DB         7577
eess.AS       6817
math.CO       5876
cs.CC         5806
Name: count, dtype: int64

In [27]:
# Categorías principales más frecuentes dentro de Ciencias Exactas
df_exact["main_category"].value_counts().head(30)

main_category
hep-ph                141480
quant-ph              128185
hep-th                112079
astro-ph               94246
gr-qc                  70608
cond-mat.mtrl-sci      69606
cond-mat.mes-hall      68640
math.AP                56477
astro-ph.GA            53799
math.CO                52165
cond-mat.str-el        52082
astro-ph.SR            47682
cs.LG                  46106
astro-ph.HE            45299
astro-ph.CO            44302
math.PR                43383
cond-mat.stat-mech     43284
math.AG                39575
cs.IT                  39054
math.OC                38011
math.NT                36663
nucl-th                35404
physics.optics         34854
math.NA                34305
math-ph                33839
cond-mat.supr-con      33763
math.DG                31918
cond-mat.soft          31361
astro-ph.EP            25892
math.DS                25387
Name: count, dtype: int64

In [28]:
# ============================================================
# 13. LIMPIEZA DEL SUBDATASET DE COMPUTER SCIENCE
# ============================================================
# Nos quedamos con columnas clave
cs_columns = ["id", "title", "abstract", "authors", "categories", "main_category", "general_area"]

df_cs_clean = df_cs[cs_columns].copy()

# Opcional: quitar filas sin título o sin abstract
df_cs_clean = df_cs_clean[
    df_cs_clean["title"].notna() &
    df_cs_clean["abstract"].notna() &
    (df_cs_clean["title"].str.strip() != "") &
    (df_cs_clean["abstract"].str.strip() != "")
].copy()

print("Dimensiones del dataset limpio de CS:", df_cs_clean.shape)
df_cs_clean.head()

Dimensiones del dataset limpio de CS: (892992, 7)


,id,title,abstract,authors,categories,main_category,general_area
1,0704.0002,Sparsity-certifying Graph Decompositions,"We describe a new algorithm, the $(k,\ell)$-pebble game with colors, and use\nit obtain a characterization of the family of $(k,\ell)$-sparse graphs and\nalgorithmic solutions to a family of pro...",Ileana Streinu and Louis Theran,math.CO cs.CG,math.CO,Computer Science
45,0704.0046,A limit relation for entropy and channel capacity per unit cost,"In a quantum mechanical model, Diosi, Feldmann and Kosloff arrived at a\nconjecture stating that the limit of the entropy of certain mixtures is the\nrelative entropy as system size goes to infi...","I. Csiszar, F. Hiai and D. Petz",quant-ph cs.IT math.IT,quant-ph,Computer Science
46,0704.0047,Intelligent location of simultaneously active acoustic emission sources:\n Part I,"The intelligent acoustic emission locator is described in Part I, while Part\nII discusses blind source separation, time delay estimation and location of two\nsimultaneously active continuous ac...",T. Kosel and I. Grabec,cs.NE cs.AI,cs.NE,Computer Science
49,0704.0050,Intelligent location of simultaneously active acoustic emission sources:\n Part II,"Part I describes an intelligent acoustic emission locator, while Part II\ndiscusses blind source separation, time delay estimation and location of two\ncontinuous acoustic emission sources.\n A...",T. Kosel and I. Grabec,cs.NE cs.AI,cs.NE,Computer Science
61,0704.0062,On-line Viterbi Algorithm and Its Relationship to Random Walks,"In this paper, we introduce the on-line Viterbi algorithm for decoding hidden\nMarkov models (HMMs) in much smaller than linear space. Our analysis on\ntwo-state HMMs suggests that the expected ...","Rastislav \v{S}r\'amek, Bro\v{n}a Brejov\'a, Tom\'a\v{s} Vina\v{r}",cs.DS,cs.DS,Computer Science


In [36]:
# ============================================================
# 14. GUARDAR EL DATASET APARTE DE COMPUTER SCIENCE
# ============================================================
output_csv = r"C:\Users\Aaron\Documents\arxiv_cs_dataset_exploration\dataset\arxiv_computer_science.csv"
output_parquet = r"C:\Users\Aaron\Documents\arxiv_cs_dataset_exploration\dataset\arxiv_computer_science.parquet"

df_cs_clean.to_csv(output_csv, index=False, encoding="utf-8")
df_cs_clean.to_parquet(output_parquet, index=False)

print("Archivo CSV guardado en:", output_csv)
print("Archivo Parquet guardado en:", output_parquet)

ArrowKeyError: A type extension with name pandas.period already defined

In [35]:
pip install --upgrade pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# ============================================================
# 15. REPORTE FINAL
# ============================================================
print("========== REPORTE FINAL ==========")
print("Total de registros cargados:", len(df))
print("Total de categorías distintas:", all_categories_df['category'].nunique())
print("Total de papers de Computer Science:", len(df_cs_clean))
print("\nCategorías principales más frecuentes en CS:")
display(df_cs_clean["main_category"].value_counts().head(20))